In [69]:
from Association_Rule_Mining import insert_optimized_parameter, optimize_param, calculate_all_periods, create_optimized_dataset, normalize_values, perform_PCA, perform_LASSO, extend_time_columns, filter_apriori, create_tweet_dataset, create_continous_dataset, create_categorize_dataset, perform_apriori, create_triple_barrier_labeling
import pandas as pd

In [70]:
# #BTC
# optimized_df = create_optimized_dataset('BTC-USD')
# optimized_df["trade_profitable"] = (optimized_df["close"].shift(-1) > optimized_df["close"]).astype(int)
market = 'AMZN'
STARTING_DATE = "2018-01-01"
ENDING_DATE ="2020-01-01"

continous_df = create_continous_dataset(market, starting_date=STARTING_DATE, ending_date=ENDING_DATE)
continous_df_with_tbl = create_triple_barrier_labeling(continous_df)
tweets_df = create_tweet_dataset("Datasets/investing_classified_sentiments" if market != "BTC-USD" else "Datasets/btc_classified_sentiments") #"Datasets/investing_classified_sentiments"
merged_df = pd.concat([continous_df_with_tbl, tweets_df], axis=1).dropna()
extended = merged_df #extend_time_columns(merged_df, skip_cols=["signals"], t=7)
categorized_for_trade_profitable = create_categorize_dataset(extended, suffix_vals=['bearish', 'Bearish', 'bullish', 'Bullish', '1.0', '1', '0.0', '0', '-1', '-1.0'], skip_cols=["next_day_label", "signals", "previous_label"])

[*********************100%***********************]  1 of 1 completed


Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Optimizing:   0%|          | 0/5 [00:00<?, ?it/s]

Label distribution: 
2.0    170
0.0    110
1.0     93
Name: label, dtype: int64 0    1.983490
1    6.107849
2    3.879653
Name: sharpe_ratio, dtype: float64


In [71]:
categorized_for_trade_profitable.columns

Index(['trade_profitable_0', 'trade_profitable_1', 'close_increasing_0',
       'close_increasing_1', 'crossover_0', 'crossover_1',
       'Momentum_Increasing_0', 'MACD_Increasing_0', 'MACD_Increasing_1',
       'volatility_label_bearish', 'volatility_label_bullish', 'RSI_bearish',
       'RSI_bullish', 'ROC_bearish', 'ROC_bullish',
       'sen_compound_increasing_0', 'sen_compound_increasing_1',
       'sen_positive_increasing_0', 'sen_positive_increasing_1',
       'sen_negative_increasing_0', 'sen_negative_increasing_1'],
      dtype='object')

In [77]:
pd.set_option('display.max_colwidth', None)
all = filter_apriori(perform_apriori(categorized_for_trade_profitable, min_support = 0.01, min_confidence=0.1), min_lift=0.1, target_label = 'trade_profitable_1', min_antecedents=3, max_antecedents=5)
all.drop_duplicates(subset=['antecedents'])

,antecedents,support,confidence,lift,target_label
0,"(sen_negative_increasing_0, close_increasing_0, sen_compound_increasing_0, crossover_0)",0.012308,1.000000,1.815642,trade_profitable_1
1,"(sen_negative_increasing_1, sen_positive_increasing_1, RSI_bearish, crossover_0)",0.012308,1.000000,1.815642,trade_profitable_1
2,"(volatility_label_bullish, crossover_1, ROC_bearish, sen_positive_increasing_1)",0.021538,0.875000,1.588687,trade_profitable_1
3,"(close_increasing_1, MACD_Increasing_0, volatility_label_bearish, crossover_0)",0.015385,0.833333,1.513035,trade_profitable_1
4,"(volatility_label_bullish, crossover_1, MACD_Increasing_0, sen_positive_increasing_1)",0.027692,0.818182,1.485526,trade_profitable_1
...,...,...,...,...,...
2021,"(volatility_label_bullish, sen_negative_increasing_1, sen_compound_increasing_0, crossover_0)",0.012308,0.307692,0.558659,trade_profitable_1
2022,"(volatility_label_bullish, close_increasing_0, crossover_0)",0.015385,0.294118,0.534012,trade_profitable_1
2023,"(volatility_label_bullish, close_increasing_0, Momentum_Increasing_0, crossover_0)",0.015385,0.294118,0.534012,trade_profitable_1
2024,"(volatility_label_bullish, crossover_0, sen_negative_increasing_1)",0.012308,0.266667,0.484171,trade_profitable_1


In [73]:
continous_df = merged_df.copy()
for col in merged_df.columns:
    if merged_df[col].nunique() <= 3:
        continous_df = continous_df.drop(col, axis=1)
continous_df = continous_df.dropna()

In [74]:
continous_df

,close,high,low,open,volume,close_pct_change,SMA_50,SMA_10,Momentum_Val,MACD_Val,volatility,lower_barriers,upper_barriers,tp_stop,sl_stop,RSI_Value,ROC_Value,bearish_mean,bearish_min,bearish_max,bearish_std,neutral_mean,neutral_min,neutral_max,neutral_std,bullish_mean,bullish_min,bullish_max,bullish_std,sen_compound
2018-04-20,76.374496,78.059998,75.804497,78.059998,110832000.0,-0.018896,74.42085,73.531049,71.539497,0.312283,0.023439,73.553122,78.663099,0.029966,0.036941,72.384350,-0.018896,0.177672,0.007633,0.951154,0.236055,0.107515,0.007005,0.973370,0.246170,0.714814,0.018689,0.950419,0.309230,0.537142
2018-04-23,75.892998,77.400002,75.170502,77.334503,89308000.0,-0.006304,74.58821,74.089948,72.074997,0.419446,0.024254,73.553122,78.663099,0.036500,0.030831,72.221376,-0.006304,0.165343,0.007218,0.949241,0.219993,0.088828,0.007151,0.971399,0.220571,0.745829,0.020898,0.946023,0.287663,0.580486
2018-04-24,73.004501,76.974998,72.422501,76.790001,149894000.0,-0.038060,74.70870,74.209299,75.191498,0.268204,0.032923,73.553122,78.663099,0.077510,0.000000,52.408054,-0.038060,0.117683,0.007421,0.950526,0.157704,0.090218,0.007147,0.971456,0.201801,0.792099,0.009486,0.947496,0.243709,0.674416
2018-04-25,73.008499,73.499496,70.750999,72.900002,131746000.0,0.000055,74.78264,74.374899,76.391998,0.146973,0.034913,69.185067,76.067245,0.041896,0.052370,56.586865,0.000055,0.141031,0.011869,0.947118,0.188354,0.070190,0.006772,0.949659,0.173333,0.788780,0.021096,0.947637,0.245437,0.647749
2018-04-26,75.898003,76.471001,73.925003,74.250504,176022000.0,0.039578,74.88609,74.722198,77.845497,0.280817,0.035414,69.185067,76.067245,0.002230,0.088447,64.154041,0.039578,0.105474,0.008797,0.946282,0.154616,0.065561,0.007900,0.973771,0.173340,0.828965,0.011009,0.948470,0.223482,0.723491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-09-23,89.264999,89.635002,88.365997,88.849998,58446000.0,-0.004938,92.27144,90.876199,90.391998,-0.275342,0.011396,89.875138,92.274856,0.033718,0.000000,33.797617,-0.004938,0.119745,0.009082,0.943132,0.173647,0.059962,0.006836,0.970298,0.148657,0.820294,0.020431,0.949979,0.217275,0.700549
2019-09-24,87.080498,89.785500,86.777496,89.530502,92320000.0,-0.024472,91.99206,90.481499,91.127502,-0.529703,0.014641,85.678027,88.482968,0.016105,0.016105,13.446643,-0.024472,0.144733,0.006082,0.950711,0.200105,0.094152,0.006799,0.972311,0.214452,0.761114,0.020753,0.951518,0.268063,0.616381
2019-09-25,88.416496,88.650002,86.150002,87.367996,69864000.0,0.015342,91.75049,90.208199,90.873001,-0.616377,0.013637,85.678027,88.482968,0.000752,0.030972,28.076536,0.015342,0.104600,0.007203,0.958826,0.142773,0.157387,0.007278,0.964601,0.240491,0.738013,0.018855,0.950961,0.248928,0.633413
2019-09-26,86.991997,88.168503,86.574997,88.139503,70736000.0,-0.016111,91.49830,89.689649,91.074997,-0.790895,0.012755,85.678027,88.482968,0.017139,0.015104,28.608266,-0.016111,0.108888,0.008667,0.944466,0.155463,0.176504,0.006836,0.969801,0.262247,0.714609,0.019448,0.948996,0.267761,0.605721


In [75]:
X_scaled = normalize_values(continous_df)
perform_PCA(X_scaled, y=X_scaled["close_pct_change"], top_k = 8)
perform_LASSO(X_scaled, continous_df, target="close", n = 10)

Features kept after correlation filter (12): ['close', 'volume', 'close_pct_change', 'tp_stop', 'sl_stop', 'RSI_Value', 'ROC_Value', 'bearish_mean', 'bearish_std', 'neutral_mean', 'neutral_std', 'bullish_max']
Explained variance (cumulative): [0.279 0.476 0.623 0.75  0.83 ]

PCA + Correlation selected features:
                  Correlation_with_target  Max_abs_loading
bullish_max                      0.141717         0.966930
neutral_mean                    -0.168715         0.559126
bearish_std                      0.101947         0.488967
neutral_std                     -0.213185         0.478454
ROC_Value                        1.000000         0.463534
close_pct_change                 1.000000         0.463534
close                            0.111040         0.449253
bearish_mean                     0.116816         0.446054
LASSO selected features: ['close_pct_change', 'SMA_50', 'Momentum_Val', 'MACD_Val', 'tp_stop', 'sl_stop', 'ROC_Value', 'neutral_min', 'neutral_max', 'neutra